> **Release dependency:** the original `clinical_review/` helper package was absent from the supplied source tree. This notebook is retained for documentation but cannot run until those original helpers are restored. See `docs/REPRODUCIBILITY.md`. No completed reviewer data are distributed.

# BR08 — One clinician: initial report assessment and discrepancy adjudication

Use the same review directory as BR07. No model loading. Ratings are saved one at a time. Initial ratings freeze before adjudication; original disagreements are never overwritten. The SAME clinician performs a separate second pass for disagreements, errors, omissions, uncertainty or uninterpretable ECGs. A confirmed disagreement remains a report error; agreement with the model is not the goal. This does not establish inter-rater reliability.

In [ ]:
from pathlib import Path
import sys
EXP = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
            if (p / 'config.example.json').is_file() and (p / 'common.py').is_file()), None)
if EXP is None:
    raise FileNotFoundError('Start Jupyter inside graph_reasoning/experiment02')
sys.path.insert(0, str(EXP))
from common import load_config, parent_setup, read
# Use the EXACT completed BR04B child config. Change this for later cohorts.
CONFIG_PATH = Path('/path/to/repository/outputs/real_run_001/'
                   'experiment02_v1/context_fit/v1/internal_val/config.json')
if not CONFIG_PATH.is_file():
    raise FileNotFoundError('Set CONFIG_PATH to the actual BR04B context-fit child config: ' + str(CONFIG_PATH))
CFG = load_config(CONFIG_PATH)
parent_setup(CFG)
RUN = Path(CFG['run_dir'])
REVIEW_VERSION = 'calibration_v1'  # Same value in BR07, BR08 and BR09. New version for main review.
REVIEW_DIR = RUN / 'clinical_review' / CFG['cohort'] / REVIEW_VERSION
from clinical_review.workflow import progress
print('Cohort:', CFG['cohort'], '| review:', REVIEW_DIR)
if CFG['cohort'] == 'internal_val':
    print('Validation review is calibration/exploratory, not held-out clinical validation.')


In [ ]:
from clinical_review.workflow import release_reports, save_rating, prepare_adjudication, save_adjudication
print(progress(REVIEW_DIR))
# Stops if any independent ECG interpretation is missing. Safe to rerun unchanged.
print(release_reports(REVIEW_DIR))


## Initial review — repeat display and save for every report

Do not open PRIVATE or automatic scores. Review reports independently rather than comparing arms side by side. Use the ECG plus your frozen interpretation. Clinically significant errors/omissions are findings potentially affecting interpretation or management. Explicit zero means assessed and absent; None means unassessed. Explain every discrepancy, uncertainty or interpretability limitation.

In [ ]:
from IPython.display import display, Image
reports = read(REVIEW_DIR / 'PUBLIC/reports.json')
remaining = [r for r in reports if not (REVIEW_DIR / 'PUBLIC/rating_drafts' / (r['review_id'] + '.json')).exists()]
ITEM = remaining[0] if remaining else None
if ITEM is None:
    print('All initial ratings saved. Proceed to the adjudication submission cell.')
else:
    print('Anonymous report:', ITEM['review_id'])
    display(Image(filename=str(REVIEW_DIR / 'PUBLIC' / ITEM['image'])))
    references = {r['case_id']: r for r in read(REVIEW_DIR / 'PUBLIC/reference_interpretations.json')}
    print('Your prior independent interpretation:', references[ITEM['case_id']])
    print('Generated report:', ITEM['generated_report'])


In [ ]:
# HUMAN ENTRY ONLY. Enter all fields; no default clinical scores.
SAVE_INITIAL_RATING = False
RATING = dict(
    reviewer_id='ECG_reviewer_01', agreement=None,  # agree / disagree / uncertain
    clinically_significant_error_count=None,
    clinically_significant_omission_count=None,
    minor_error_count=None,
    overall_quality_1_to_5=None,  # 1 unusable; 2 major; 3 moderate; 4 minor corrections; 5 acceptable unchanged
    uninterpretable_image=None,  # boolean
    discrepancy_details='', comments='')
if SAVE_INITIAL_RATING:
    if ITEM is None: raise ValueError('No current report')
    save_rating(REVIEW_DIR, ITEM['review_id'], **RATING)
    print('Saved:', ITEM['review_id'], '| rerun the display cell for the next report')
else:
    print('No rating saved. Fill the fields from the clinician assessment before enabling save.')


## Freeze the initial assessment and create the adjudication queue

Run only after all initial reports have been assessed. Missing assessments stop submission. After submission, use adjudication to record corrections; the initial judgments remain immutable. Prefer a later review session for the following pass. Items flagged by image uncertainty also require a decision.

In [ ]:
print(prepare_adjudication(REVIEW_DIR))
print(progress(REVIEW_DIR))


In [ ]:
queue = read(REVIEW_DIR / 'PUBLIC/adjudication_queue.json')
pending = [r for r in queue if not (REVIEW_DIR / 'PUBLIC/adjudications' / (r['review_id'] + '.json')).exists()]
ADJ_ITEM = pending[0] if pending else None
if ADJ_ITEM is None:
    print('No pending adjudications. Continue to BR09.')
else:
    print('Adjudication:', ADJ_ITEM['review_id'])
    display(Image(filename=str(REVIEW_DIR / 'PUBLIC' / ADJ_ITEM['image'])))
    print('Independent interpretation:', ADJ_ITEM['independent_interpretation'])
    print('Generated report:', ADJ_ITEM['generated_report'])
    print('Original rating:', ADJ_ITEM['initial_rating'])


In [ ]:
# SAME clinician, separate assessment. Never revise merely to agree with the model.
SAVE_ADJUDICATION = False
DECISION = None  # retain_initial / revise_initial / unresolved
REASON = ''  # Required clinical rationale, including any correction to the initial interpretation.
EVIDENCE_REVIEWED = ''  # Describe query ECG features/source actually rechecked.
FINAL_RATING = None  # retain_initial: copied unchanged; unresolved: None.
# For revise_initial ONLY, supply all rating fields with explicit clinician-assigned values.
# FINAL_RATING = dict(reviewer_id=..., agreement=..., clinically_significant_error_count=...,
#    clinically_significant_omission_count=..., minor_error_count=...,
#    overall_quality_1_to_5=..., uninterpretable_image=..., discrepancy_details=..., comments=...)
if SAVE_ADJUDICATION:
    if ADJ_ITEM is None: raise ValueError('No pending adjudication')
    save_adjudication(REVIEW_DIR, ADJ_ITEM['review_id'],
        reviewer_id=ADJ_ITEM['initial_rating']['reviewer_id'], decision=DECISION,
        reason=REASON, evidence_reviewed=EVIDENCE_REVIEWED, final_rating=FINAL_RATING)
    print('Submitted:', ADJ_ITEM['review_id'], '| rerun the display cell for the next item')
else:
    print('No adjudication saved. Enter the actual second-pass assessment before enabling save.')
print(progress(REVIEW_DIR))


Submitted adjudications cannot be overwritten. A confirmed report discrepancy is a valid adjudication outcome. An unresolved item remains missing for definitive clinical scoring; BR09 reports it explicitly with bounds. No second clinician is required by this workflow. Human input must remain genuine; never batch-fill the forms with model predictions.